### 0) importing libraries

In [118]:
from collections import OrderedDict,Counter
import ast
import math
import time
import joblib

# plotting
import seaborn as sb
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# manage dataframe
import pandas as pd

# manage graph structures
import networkx as nx
import networkx.algorithms.community as nxcom

# statistic, ML, dimentionality reduction packages
from scipy.interpolate import interp1d
from sklearn.metrics.pairwise import cosine_distances,pairwise_distances
from sklearn import preprocessing
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.manifold import TSNE, MDS, SpectralEmbedding
from sklearn.cluster import AgglomerativeClustering,SpectralClustering,DBSCAN
import umap.umap_ as umap
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import matthews_corrcoef,f1_score,precision_score,recall_score,confusion_matrix,roc_auc_score,brier_score_loss,roc_curve,precision_recall_curve,auc
from sklearn.model_selection import GridSearchCV

# handling mixed type distances
import gower as gw

# annoying warning
import warnings
warnings.filterwarnings('ignore')

# take a look at the Kepler Mapper package!
import kmapper as km
from kmapper.plotlyviz import *

https://kepler-mapper.scikit-tda.org/en/latest/index.html

#### ---------------------------------------------------------------------------------------------------------------------------------------------------------

### 1) Distance matrix computation 

In [27]:
def distance_matrix(patient_id, Y_class, dataset,continue_features, visualize, distance_matrix_path):
    """
    Function for distance matrix calculation, according to the features' type
    INPUT:
        - patient_id:           (pandas Series) dataset's column indicating the samples IDs
        - Y_class:              (pandas Series) dataset's column indicating the binarized output (phenotypes)
        - dataset               (pandas DataFrame) dataset for which calculate the distance
        - continue_features     (list) list containing the dataset's numerical features
        - visualize             (boolean) flag to specify that the distance matrix will be plotted with a heatmap
        - distance_matrix_path  (string)  path in which the distance matrix will be saved as .npy
    OUTPUT:
        - distance_matrix       (numpy ndarray) patients distance matrix
    """
    
    # filter the dataset in order to delete the IDs and the output columns
    dataset_features = dataset.loc[:, ~dataset.columns.isin([patient_id.name,Y_class.name])]
    
    # find all the categorical variables
    bool_categorical = []
    for features in list(dataset_features.columns):
        if features in continue_features:
            bool_categorical.append(False)
        else:
            bool_categorical.append(True)
    
    # only categorical variables -> Jaccard distance
    if sum(bool_categorical) == 0:
        print("only categorical variables -> Jaccard distance")
        
        # make the categorical variables as dummies
        df_categorical_dummies = pd.get_dummies(dataset_features.astype(str),drop_first=True)
        distance_matrix = pairwise_distances(df_categorical_dummies.values, metric = "jaccard")
    
    
    # both categorical and continue variables -> Gower distance
    elif sum(bool_categorical)>0 and sum(bool_categorical)!=len(bool_categorical):
        print("both categorical and continue variables -> Gower distance")

        dataset_features[continue_features] = dataset_features[continue_features].astype(float)
        
        # the gower package already creates dummy variables and standardize numerical variables
        distance_matrix = gw.gower_matrix(dataset_features,cat_features=bool_categorical)
        
        
    # only numerical variables -> cosine distance 
    elif sum(bool_categorical)==len(bool_categorical):
        print("only numerical variables -> cosine distance")
        
        # scaling the numerical features in [0,1] range
        X = dataset_features.values
        min_max_scaler = preprocessing.MinMaxScaler()
        X_scaled = min_max_scaler.fit_transform(X)
        dataset_scaled= pd.DataFrame(X_scaled, columns = dataset_features.columns)

        distance_matrix = cosine_distances(dataset_scaled,dataset_scaled)

    # visualize distance matrix as a heatmap
    if visualize:
        fig = plt.figure()
        sb.heatmap(distance_matrix)
        
    # save the distance matrix as .npy file
    np.save(distance_matrix_path,distance_matrix,allow_pickle=False)
    
    return distance_matrix

In [ ]:
# insert the dataset path and read the dataset
dataset_experiment = pd.read_excel("") 

# inspect the dataset features
dataset_experiment.head()


In [ ]:
# create a list with the dataset numerical variables  
continue_features =  ['X1']

# run the distance matrix function, by correctly specifying the inputs and plot it as a heatmap
distance_matrix = distance_matrix(dataset_experiment['Patient_ID'],
                                  dataset_experiment['Y'],
                                  dataset_experiment,
                                  continue_features,
                                  True,
                                  "./distance_matrix.npy")

#### ---------------------------------------------------------------------------------------------------------------------------------------------------------

### 2.1) TDA Mapper pipeline - first step of the grid search

In [86]:
def entropy_count(scomplex, initial_class):
    """
    Function for computing the graph entropy.
    INPUT:
       - scomplex:     (dictionary) simplicial complex resulting from the application of KeplerMapper
       - initial_class: (pandas Series) dataset's column indicating the initial phenotypes
    OUTPUT:
       - entropy: (float) graph entropy (weighted mean average nodes entropy)
    """
    
    initial_class_ = initial_class.copy(True)

    kmgraph = get_mapper_graph(scomplex)
    # assign to node['custom_tooltips']  the node label
    for node in kmgraph[0]['nodes']:
        node['label'] = initial_class_[scomplex['nodes'][node['name']]]
    entropy_node = {}

    label_values = list(initial_class_.value_counts().index)
    label_values.sort()

    # extract nodes properties such as size and how many samples for each class
    for j, node in enumerate(kmgraph[0]['nodes']):   
        dimCluster = node['cluster']['size']
            
        data_bin = node['label']

        dimData_dict = {}
        for label in label_values:
            dimData_dict[label] = len(data_bin.loc[data_bin==label])

        # compute the entropy by looking at each label frequency
        H_node = 0.0
        for k,v in dimData_dict.items():
            # only if the label is present in the node!
            if v != 0:
                H_node+=  -(v/dimCluster)*math.log2(v/dimCluster)
                
        entropy_node[j] = [dimCluster, H_node]
        
    # compute the average weighted entropy -> graph entropy        
    sumBinEntropies = 0
    numberData = 0
    for j, node in enumerate(entropy_node):
        sumBinEntropies += (entropy_node[j][0]*entropy_node[j][1])
        numberData += entropy_node[j][0]
    graph_entropy = sumBinEntropies/numberData
    
    return graph_entropy


def do_hist_counter(scomplex):
    """
    Function to calculate the nodes size distribution.
    INPUT:
       - scomplex: (dictionary) simplicial complex resulting from the application of KeplerMapper
    OUTPUT:
       - counter:  (list) list of integer containing the distribution of the nodes size
       - n_nodes:  (int)  number of graph nodes
       - n_edges:  (int)  number of graph edges
       - n_unique: (int)  number of unique samples (patients) that results in the simplicial commplex
    """    
    kmgraph,  mapper_summary, colorf_distribution = get_mapper_graph(scomplex, color_function_name='Distance to x-min')

    n_nodes = mapper_summary['n_nodes']
    n_edges = mapper_summary['n_edges']
    n_unique = mapper_summary['n_unique']
    
    counter = list()
    for j, node in enumerate(kmgraph['nodes']):
        counter.append(node["cluster"]["size"])

    return counter,n_nodes,n_edges,n_unique

def estimate_dbscan_params(data, smooth_n=1000):
    """
    Estimate optimal DBSCAN parameters (`eps` and `minPts`) using the elbow method and log rule.
    
    Parameters:
    - datadf: (pandas DataFrame) dataset of interest
    - smooth_n: Number of interpolation points for smoother elbow curve (default: 1000)
    
    Returns:
    - eps_estimated: float, estimated `eps` value
    - minPts: int, estimated `minPts` value
    """

    n_samples = data.shape[0]
    minPts = int(np.round(np.log(n_samples)))

    # Use k+1 because kNN includes the point itself
    nbrs = NearestNeighbors(n_neighbors=minPts + 1).fit(data)
    distances, _ = nbrs.kneighbors(data)

    # Get the k-th nearest distance (excluding self-distance)
    k_distances = np.sort(distances[:, minPts])

    # Interpolate to smooth the curve
    x_vals = np.linspace(0, len(k_distances) - 1, smooth_n)
    interpolator = interp1d(np.arange(len(k_distances)), k_distances, kind='linear')
    y_vals = interpolator(x_vals)

    # First and second derivative for elbow detection
    dy = np.gradient(y_vals, x_vals)
    d2y = np.gradient(dy, x_vals)

    # Find the elbow as the maximum second derivative
    elbow_idx = np.argmax(np.abs(d2y))
    eps_estimated = y_vals[elbow_idx]

    if eps_estimated==0.0:
        eps_estimated=0.1
    
    return eps_estimated, minPts

In [84]:
def lenses_and_hyperparameters_greedy_search(distance_matrix,lens_names, projection_dimension,
                                              metric_mds, n_init_mds,max_iter_mds,eps_mds,
                                              n_neighbor_isomap,
                                              perplexities,learning_rates,n_iters,
                                              min_dists,n_neighbors,      
                                              n_cubes,perc_overlap,cluster_methods,
                                              rnd,initial_class, biological_features, path_grid_search_results,
                                              flag_remove_duplicate_nodes):
    """
    Function for the lens functions and their hyperparameters grid search. 
    The execution time (in minutes) is printed at the end.
    INPUT:
       - distance_matrix:                            (numpy ndarray) patients distance matrix
       - lens_names:                                 (list) list with the lens functions to include in the grid search
       - projection_dimension:                       (int) integer that indicates the number of dimensios of the projected space

       MDS parameters as sklearn
       - [metric_mds (list of bool); n_init_mds (list of int); max_iter_mds (list of int); eps_mds (list of float)]
        ISOMAP parameters as sklearn
       - [n_neighbor_isomap (list of int)]
       t-SNE parameters as sklearn
       - [ perplexities (list of int) ; learning_rates (list of int) ; n_iters (list of int) ]
       UMAP parameters as umap-learn
       - [ min_dists (list of float) ; n_neighbors (list of int) ]
       
       - n_cubes:                                    (list of int) list of number of hypercubes for each project dimensions, i.e. resolution
       - perc_overlap:                               (list of float) list of percentages of overalap, i.e. gain
       - cluster_methods:                            (list of sklearn cluster methods) list of cluster methods to use in the grid search
       
       - rnd:                                        (int) seed for reproducible output
       - initial_class:                              (pandas Series) dataset's column indicating the initial phenotypes
       - biological_features:                        (pandas DataFrame) a DataFrame with two columns to use as biological lens.
                                                     The first column will be used as projection with l2 if specifying "Bio_l2" lens;  
                                                     Both columns will be used as projections if specifying "Bio_bio" lens.
       - path_grid_search_results:                   (string) path in which the output dataframe will be saved
       - flag_remove_duplicate_nodes                 (boolean) if eliminate duplicated node in the mapper
    OUTPUT:
       - grid_search_results_df:                     (pandas DataFrame) report of the grid search 
    """
    
    # start by creating an empy dataframe. This will be appended with the grid search combinations of parameters and graph statistics
    grid_search_results_df = pd.DataFrame(columns = ["lens_name","parameters_combination",
                            "n_nodes","n_edges","n_unique","graph_entropy",
                            "node_size_distribution_values", "mean_size",
                            "nodes_degree", "mean_degree", "density"])
    st = time.time()
    mapper = km.KeplerMapper()    

    # cicle on all the lens functions specified in the list. 
    # For each functions all the parameters combination are tried, a projection lens in created and the dataset is projected
    # then the projection are used by the 'Mapper_parameters_greedy_search' functions to create the graph
    
    for lens in lens_names:
        
        print("lens: "+lens)

        if lens=="PCA":
            projection = mapper.project(distance_matrix, projection=PCA(n_components=projection_dimension, random_state=rnd), distance_matrix = None, scaler = None)
            
            parameters_combination_string = "-" 
            grid_search_results_df = Mapper_parameters_greedy_search(distance_matrix,mapper,projection,
                                                         n_cubes,perc_overlap,cluster_methods, 
                                                         lens, parameters_combination_string,grid_search_results_df,initial_class,flag_remove_duplicate_nodes,rnd)

        elif lens=="MDS":
            for m in metric_mds:
                for n in n_init_mds:
                    for maxi in max_iter_mds:
                        for eps in eps_mds:
                            projection = mapper.project(distance_matrix, projection=MDS(n_components=projection_dimension, random_state=rnd, metric='precomputed',
                                                                                        metric_mds=m, max_iter = maxi, eps_mds = eps, n_init = n), distance_matrix = None, scaler = None)
                            parameters_combination_string = str("metric: "+ str(m)+"; n_init: "+ str(n) +"; max_iter: "+ str(maxi)+"; eps: "+ str(eps)) 
                            grid_search_results_df = Mapper_parameters_greedy_search(distance_matrix,mapper,projection,
                                                         n_cubes,perc_overlap,cluster_methods, 
                                                         lens, parameters_combination_string,grid_search_results_df,initial_class,flag_remove_duplicate_nodes,rnd)

        elif lens=="Isomap":
            for n in n_neighbor_isomap:
                projection = mapper.project(distance_matrix, projection=Isomap(n_components=projection_dimension, path_method="D",metric="precomputed",n_neighbors=n), distance_matrix = None, scaler = None)
                
                parameters_combination_string = str("n_neighbor: "+ str(n)) 
                grid_search_results_df = Mapper_parameters_greedy_search(distance_matrix,mapper,projection,
                                                             n_cubes,perc_overlap,cluster_methods, 
                                                             lens, parameters_combination_string,grid_search_results_df,initial_class,flag_remove_duplicate_nodes,rnd)
                
        elif lens=="tSNE":
            for p in perplexities:
                for l in learning_rates:
                    for i in n_iters:
                        print("perplexity: "+ str(p)+"; learning_rates: "+ str(l) +"; n_iters: "+ str(i)   )
                        projection = mapper.project(distance_matrix, projection= TSNE(n_components=projection_dimension, random_state=rnd, perplexity = p , learning_rate= l , max_iter= i,init="random",metric="precomputed"),distance_matrix = None, scaler = None)

                        parameters_combination_string = str("perplexity: "+ str(p)+"; learning_rates: "+ str(l) +"; n_iters: "+ str(i)) 
                        grid_search_results_df = Mapper_parameters_greedy_search(distance_matrix,mapper,projection,
                                                         n_cubes,perc_overlap,cluster_methods, 
                                                         lens, parameters_combination_string,grid_search_results_df,initial_class,flag_remove_duplicate_nodes,rnd)
                
    
                        
        elif lens=="UMAP": 
            for n in n_neighbors:
                for d in min_dists:
                    print("min_dists: "+ str(d)+"; n_neighbors: "+ str(n))
                    projection = mapper.project(distance_matrix, projection= umap.UMAP(n_components=projection_dimension, random_state=rnd, n_neighbors= n, min_dist=d,metric="precomputed"),distance_matrix = None, scaler = None)

                    parameters_combination_string = str("min_dists: "+ str(d)+"; n_neighbors: "+ str(n)) 
                    grid_search_results_df = Mapper_parameters_greedy_search(distance_matrix,mapper,projection,
                                                         n_cubes,perc_overlap,cluster_methods, 
                                                         lens, parameters_combination_string,grid_search_results_df,initial_class,flag_remove_duplicate_nodes,rnd)
    
        elif lens=="Bio_l2":
            
            print(biological_features.iloc[:,0].name + "l2 norm")
            
            parameters_combination_string = biological_features.iloc[:,0].name
            
            bio_feature = np.array(biological_features.iloc[:,0]).reshape((biological_features.iloc[:,0].shape[0], 1))
            lens2 = mapper.fit_transform(distance_matrix, projection="l2norm", distance_matrix = None, scaler = None)
            projection = np.c_[lens2, bio_feature]
                
            grid_search_results_df = Mapper_parameters_greedy_search(distance_matrix,mapper,projection, control_value,
                                                n_cubes,perc_overlap,cluster_methods, 
                                                lens, parameters_combination_string,grid_search_results_df,initial_class,flag_remove_duplicate_nodes,rnd)
    
    
    grid_search_results_df.to_excel(path_grid_search_results,index=False)
    
    et = time.time()
    elapsed_time = (et - st)/60
    print('Execution time:', elapsed_time, 'minutes')
    
    return grid_search_results_df

def Mapper_parameters_greedy_search(distance_matrix,mapper,projection,
                         n_cubes,perc_overlap,cluster_methods,
                        lens, parameters_string_combination, grid_search_results_df,initial_class,flag_remove_duplicate_nodes, rnd):
    """
    Function for the Mapper parameters (resolution, gain and clustering algorithm) grid search. 
    INPUT: 
       - distance_matrix:               (numpy ndarray) patients distance matrix
       - mapper:                        (KeplerMapper object)
       - projection:                    (numpy ndarray) projections obtained with KeplerMapper method 'mapper.project'
       - n_cubes:                       (list of int) list of number of hypercubes for each project dimensions, i.e. resolution
       - perc_overlap:                  (list of float) list of percentages of overalap, i.e. gain
       - cluster_methods:               (list of sklearn cluster methods) list of cluster methods to use in the grid search
       - lens:                          (string) lens functions name
       - parameters_string_combination: (string) string containing the lens functions hyperparameters combination
       - grid_search_results_df:        (pandas DataFrame) report of the grid search 
       - initial_class:                 (pandas Series) dataset's column indicating the initial phenotypes
       - flag_remove_duplicate_nodes:   (boolean) flag used to specify if the KeplerMapper will create a graph in which duplicates nodes are excluded
       - rnd:                           (int) seed for reproducible output
    OUTPUT:
       - grid_search_results_df:        (pandas DataFrame) report of the grid search
    """
    # for each combination of mapper parameters apply the cover with the KeplerMapper method '.Cover', 
    # then compute the graph entropy, distribution of nodes size and graph statistics
    for cub in n_cubes:
        for over in perc_overlap:
            for method in cluster_methods:
                print("n_cubes: "+ str(cub)+  ";perc overlap: "+  str(over)   +"; cluster method: "+ str(method))

                if method == "DBSCAN":

                    # estimate dbscan parameters with the elbow method
                    eps_val, minPoints_val = estimate_dbscan_params(projection)
                    cluster_method_to_use = DBSCAN(metric="precomputed",min_samples=minPoints_val,eps=eps_val,n_jobs=-1)

                    scomplex = mapper.map(projection, distance_matrix, 
                    cover=km.Cover(n_cubes=cub, perc_overlap=over), clusterer=cluster_method_to_use,  precomputed=True,
                                         remove_duplicate_nodes = flag_remove_duplicate_nodes)

                elif method == "agglomerative_complete":
                    cluster_method_to_use = AgglomerativeClustering(metric='precomputed', linkage='complete',n_clusters=2)

                    scomplex = mapper.map(projection, distance_matrix, 
                    cover=km.Cover(n_cubes=cub, perc_overlap=over), clusterer=cluster_method_to_use,  precomputed=True,
                                         remove_duplicate_nodes = flag_remove_duplicate_nodes)

                elif method == "agglomerative_average":
                    cluster_method_to_use =  AgglomerativeClustering(metric='precomputed', linkage='average',n_clusters=2)

                    scomplex = mapper.map(projection, distance_matrix, 
                    cover=km.Cover(n_cubes=cub, perc_overlap=over), clusterer=cluster_method_to_use,  precomputed=True,
                                         remove_duplicate_nodes = flag_remove_duplicate_nodes)

                elif method == "agglomerative_single":
                    cluster_method_to_use = AgglomerativeClustering(metric='precomputed', linkage='single',n_clusters=2)

                    scomplex = mapper.map(projection, distance_matrix, 
                    cover=km.Cover(n_cubes=cub, perc_overlap=over), clusterer=cluster_method_to_use,  precomputed=True,
                                         remove_duplicate_nodes = flag_remove_duplicate_nodes)

                elif method == "kmedoids": 

                    cluster_method_to_use = KMedoids(metric="precomputed",n_clusters=2, init = "heuristic", random_state = rnd)

                    scomplex = mapper.map(projection, distance_matrix, 
                    cover=km.Cover(n_cubes=cub, perc_overlap=over), clusterer=cluster_method_to_use,  precomputed=True,
                                         remove_duplicate_nodes = flag_remove_duplicate_nodes)

                elif method == "spectral_clustering": 

                    cluster_method_to_use = SpectralClustering(n_clusters=2,assign_labels='kmeans', affinity="precomputed",n_jobs=-1, random_state = rnd)

                    # Note that spectral clustering works with similarities instead of distances!
                    scomplex = mapper.map(projection, 1-distance_matrix, 
                    cover=km.Cover(n_cubes=cub, perc_overlap=over), clusterer=cluster_method_to_use,  precomputed=True,
                                            remove_duplicate_nodes = flag_remove_duplicate_nodes)

                else:
                    if str(method)[0:18] == "SpectralClustering":
                        cluster_method_to_use = method
                        scomplex = mapper.map(projection, 1-distance_matrix, 
                        cover=km.Cover(n_cubes=cub, perc_overlap=over), clusterer=method,  precomputed=True,
                                         remove_duplicate_nodes = flag_remove_duplicate_nodes)
                    else:
                        cluster_method_to_use = method
                        scomplex = mapper.map(projection, distance_matrix, 
                        cover=km.Cover(n_cubes=cub, perc_overlap=over), clusterer=method,  precomputed=True,
                                             remove_duplicate_nodes = flag_remove_duplicate_nodes)
                    
                
                string_to_return_parameters_mapper = str("; n_cubes: "+ str(cub)+  ";perc overlap: "+  str(over)   +"; cluster method: "+ str(cluster_method_to_use))
                
                # compute the graph entropy
                graph_entropy = entropy_count(scomplex,initial_class)
                # compute the distribution of the nodes size
                node_size_distribution_values,n_nodes,n_edges,n_unique = do_hist_counter(scomplex)   
        
                # return a networkx graph object from the KeplerMapper simplicial complex
                G = km.adapter.to_nx(scomplex)
                
                # compute graph statistics 
                mean_size = np.mean(node_size_distribution_values)
                nodes_degree = [G.degree[n] for n in G.nodes]
                mean_degree = np.mean(nodes_degree)
                density = nx.density(G)
                
                grid_search_result_series = pd.Series({"lens_name":lens,
                            "parameters_combination":parameters_string_combination + string_to_return_parameters_mapper,
                            "n_nodes":n_nodes,"n_edges":n_edges,"n_unique":n_unique,
                            "graph_entropy":graph_entropy,
                            "node_size_distribution_values":node_size_distribution_values,
                            "mean_size":mean_size,
                            "nodes_degree":nodes_degree, "mean_degree":mean_degree,
                            "density":density})

                grid_search_results_df = pd.concat([grid_search_results_df, grid_search_result_series.to_frame().T ], ignore_index=True)
                
                
    return grid_search_results_df

In [ ]:
# define the input for the first step of the grid search
random_seed = 123
n_dimension = 2
list_lens_functions = ['PCA','tSNE','UMAP','Bio_l2','Bio_bio']

# perform the first step of the grid search by specifying the lens functions names and their hyperparameters. We fix the mapper parameters
first_step_grid_search_results_df = lenses_and_hyperparameters_greedy_search(distance_matrix, 
                                                                             list_lens_functions, 
                                                                             n_dimension, 
                                                                             [False,True],[3,4,5],[100,300,500],[1e-5, 1e-4, 1e-3, 1e-2],
                                                                             np.array([5,10,25,50,120,150,200]),
                                                                             list( np.arange(10,60,10)),list( np.arange(300,1000,100)), [300],
                                                                             np.array([0.25,0.5,0.75,0.9]),np.array([5,10,25,50,120,150,200]),
                                                                             np.array([18]),np.array([0.5]), 
                                                                             ['DBSCAN'],
                                                                             random_seed, 
                                                                             dataset_experiment['Y'], 
                                                                             dataset_experiment[['X1','X2']], 
                                                                             "./results/lens_parameters_grid_search.xlsx",
                                                                             True)

### Results

In [72]:
def read_grid_search_results(path_grid_search_results_df):
    """
    Function for reading the grid search results dataframe, saved as .xlsx.
    INPUT:
       - path_grid_search_results_df (string): path in which the grid search dataframe is saved
    OUTPUT:
       - grid_search_results_df      (pandas DataFrame): grid search results as a DataFrame
    """
    grid_search_results_df = pd.read_excel(path_grid_search_results_df)

    node_size_distribution_values = []
    nodes_degree = []

    # in order to correctly encode as list the excel cells that contains a list, we use the 'ast' package
    for index,row in grid_search_results_df.iterrows():

        node_size_distribution_values.append(ast.literal_eval(row['node_size_distribution_values']))
        nodes_degree.append(ast.literal_eval(row['nodes_degree']))

    grid_search_results_df['node_size_distribution_values'] = node_size_distribution_values
    grid_search_results_df['nodes_degree'] = nodes_degree
    
    return grid_search_results_df


def extract_parameters_from_string_and_project(lens,distance_matrix,parameters_string,projection_dimension,rnd,biological_features):
    """
    Function for extracing the hyperparameters combination from a row of the the grid search results dataframe.
    Then project the dataset with the lens and its hyperparameters combination.
    INPUT:
       - lens:                          (string) lens functions name
       - distance_matrix:               (numpy ndarray) patients distance matrix
       - parameters_string_combination: (string) string containing the lens functions hyperparameters combination
       - projection_dimension:          (int) integer that indicates the number of dimensios of the projected space
       - rnd:                           (int) seed for reproducible output
       - biological_features:           (pandas DataFrame) a DataFrame with two columns to use as biological lens.
                              The first column will be used as projection with l2 if specifying "Bio_l2" lens;  
                              Both columns will be used as projections if specifying "Bio_bio" lens.
    OUTPUT:
       - projection:                    (numpy ndarray) projections obtained with KeplerMapper method 'mapper.project'
    """
    mapper = km.KeplerMapper()
    
    print(lens)
    print(parameters_string)
               
    if lens=="PCA":
        projection = mapper.project(distance_matrix, projection=PCA(n_components=projection_dimension, random_state=rnd), distance_matrix = None, scaler = None)
    
    elif lens=="MDS":
        m = int(parameters_string.split(";")[0].split(": ")[1])
        n = int(parameters_string.split(";")[1].split(": ")[1])
        maxi = int(parameters_string.split(";")[2].split(": ")[1])
        eps = int(parameters_string.split(";")[3].split(": ")[1])
        
        projection = mapper.project(distance_matrix, projection=MDS(n_components=projection_dimension, random_state=rnd, metric='precomputed',
                                                                                        metric_mds=m, max_iter = maxi, eps_mds = eps, n_init = n), distance_matrix = None, scaler = None)                   
    
    elif lens=="Isomap":
        n = int(parameters_string.split(";")[0].split(": ")[1])
        projection = mapper.project(distance_matrix, projection=Isomap(n_components=projection_dimension, path_method="D",metric="precomputed",n_neighbors=n), distance_matrix = None, scaler = None)
                
    
    elif lens=="tSNE":
        p = int(parameters_string.split(";")[0].split(": ")[1])
        l = int(parameters_string.split(";")[1].split(": ")[1])
        i = int(parameters_string.split(";")[2].split(": ")[1])
        projection = mapper.project(distance_matrix, projection= TSNE(n_components=projection_dimension, random_state=rnd, perplexity = p , learning_rate= l , max_iter= i,init="random",metric="precomputed"),distance_matrix = None, scaler = None)
    
    elif lens=="UMAP":
        n = int(parameters_string.split(";")[1].split(": ")[1])
        d = float(parameters_string.split(";")[0].split(": ")[1])
        projection = mapper.project(distance_matrix, projection= umap.UMAP(n_components=projection_dimension, random_state=rnd, n_neighbors= n, min_dist=d,metric="precomputed"),distance_matrix = None, scaler = None)     

    elif lens=="Bio_l2":
        bio_feature = np.array(biological_features.iloc[:,0]).reshape((biological_features.iloc[:,0].shape[0], 1))
        lens2 = mapper.fit_transform(distance_matrix, projection="l2norm", distance_matrix = None, scaler = None)
        projection = np.c_[lens2, bio_feature]
        
    elif lens=="Bio_bio":
        bio_feature_1 = np.array(biological_features.iloc[:,0]).reshape((biological_features.iloc[:,0].shape[0], 1))
        bio_feature_2 = np.array(biological_features.iloc[:,1]).reshape((biological_features.iloc[:,1].shape[0], 1))
        projection = np.c_[bio_feature_1, bio_feature_2]
    
    return projection


def grid_search_results_optimal_choice(path_grid_search_results_df,initial_class,distance_matrix,projection_dimension,rnd,biological_features,annotate):
    """
    Functions that reads the grid search results and plot them. 
    In particular, the function dispaly the lens functions projections (as a scatterplot), 
    the node size and the node degree distributions (as barplots) and the graph statistics (as a table).
    In addition, graph statistics are plotted for each lens and combinations of parameters (as scatterplots).
    INPUT:
       - path_grid_search_results_df:   (string) path in which the grid search dataframe is saved
       - initial_class:                 (pandas Series) dataset's column indicating the binarized output (phenotypes)
       - distance_matrix:               (numpy ndarray) patients distance matrix
       - projection_dimension:          (int) integer that indicates the number of dimensios of the projected space
       - rnd:                           (int) seed for reproducible output
       - biological_features:           (pandas DataFrame) a DataFrame with two columns to use as biological lens.
                              The first column will be used as projection with l2 if specifying "Bio_l2" lens;  
                              Both columns will be used as projections if specifying "Bio_bio" lens.
    """
    
    grid_search_results_df = read_grid_search_results(path_grid_search_results_df)
    
    c = 0
    lens_names = set(grid_search_results_df['lens_name'])
    

    # the first figure contains the projections, the distributions of the nodes size and degree and the graph statistics   
    fig = plt.figure(figsize=(8*len(lens_names),4.5*len(lens_names)))
        
    fig_table, ax_table = plt.subplots(1,1,figsize=(8*len(lens_names),len(lens_names)-1.7))
    fig_table.patch.set_visible(False)
    ax_table.axis('off')
    ax_table.axis('tight')
        
    subfig = fig.subfigures(nrows=1, ncols=len(lens_names))
        
    columns = tuple(lens_names) 
    rows = ['N nodes', 'N edges', 'N unique samples',
            'Graph entropy', 'Mean nodes size', 'Mean nodes degree', 'Density']
        
    cell_text = []
    for lens in lens_names:
        axs = subfig[c].subplots(nrows=3, ncols=1)
            
        min_entropy = round(float(grid_search_results_df[grid_search_results_df['lens_name']==lens]['graph_entropy'].min()),4)
        min_entropy_index = grid_search_results_df[grid_search_results_df['lens_name']==lens]['graph_entropy'].idxmin()
        param_min_entropy = grid_search_results_df.loc[min_entropy_index]['parameters_combination']
        node_n = grid_search_results_df.loc[min_entropy_index]['n_nodes']
        edge_n = grid_search_results_df.loc[min_entropy_index]['n_edges']
        unique_n = grid_search_results_df.loc[min_entropy_index]['n_unique']
        node_size_distribution_values = grid_search_results_df.loc[min_entropy_index]['node_size_distribution_values']
        mean_size = grid_search_results_df.loc[min_entropy_index]['mean_size']
        nodes_degree = grid_search_results_df.loc[min_entropy_index]['nodes_degree']
        mean_degree = grid_search_results_df.loc[min_entropy_index]['mean_degree']
        density = round(float(grid_search_results_df.loc[min_entropy_index]['density']),4)
            
        cell_text.append([node_n,edge_n,unique_n,min_entropy,mean_size,mean_degree,density])
            
        projection = extract_parameters_from_string_and_project(lens,distance_matrix,param_min_entropy,projection_dimension,rnd,biological_features)

        datadf = pd.DataFrame({"Dim1":projection[:,0],"Dim2":projection[:,1],"initial class":list(initial_class)})
        counter_node_size_distribution = Counter(node_size_distribution_values)
        node_size_distribution = OrderedDict(sorted(counter_node_size_distribution.items(), key=lambda kv: kv[0]))
        bar_nodes_size = pd.DataFrame({"Node size":node_size_distribution.keys(),"Frequency":node_size_distribution.values()})
        counter_node_degree_distribution = Counter(nodes_degree)
        node_degree_distribution = OrderedDict(sorted(counter_node_degree_distribution.items(), key=lambda kv: kv[0]))
        bar_nodes_degree = pd.DataFrame({"Node degree":node_degree_distribution.keys(),"Frequency":node_degree_distribution.values()})
            
        sb.set_theme(style="white")
        sb.scatterplot(data=datadf,x="Dim1",y="Dim2",hue="initial class",s=60,ax=axs[0],legend=False)
            
        sb.set_theme(style="whitegrid")
        sb.barplot(data=bar_nodes_size,x="Node size",y="Frequency",ax=axs[1],palette=["#1f77b4"])
        sb.barplot(data=bar_nodes_degree,x="Node degree",y="Frequency",ax=axs[2],palette=["#1f77b4"])
        axs[0].set_title(lens,size=31)
        axs[0].set(ylabel=None)
        axs[0].set_xlabel("H(G) = " + str(min_entropy),size=29)
        axs[0].set_xticks([], [])
        axs[0].set_yticks([], [])
        axs[0].set_frame_on(False)

        axs[1].set_xlabel("Node size",size=22)
        axs[1].set_ylabel("Frequency",size=22)
        axs[2].set_xlabel("Node degree",size=22)
        axs[2].set_ylabel("Frequency",size=22)
            
        if c==len(lens_names)-1:
            c=0
        else:
            c+=1
        
    table = ax_table.table(cellText=cell_text,
                            rowLabels=columns,
                            colLabels=rows,
                            colWidths=[0.09,0.09,0.18,0.15,0.19,0.21 , 0.09],
                            loc='top')
    table.scale(1,4)
    table.auto_set_font_size(False)
    table.set_fontsize(18)
    
    plt.savefig("./results/lens_projections_results.png",dpi=300,bbox_inches='tight')
               
    #the second figure contains the graph statistics subplot for each combination of the lens functions parameters.
    sb.set(font_scale=1.4)
    fig, ax = plt.subplots(1,5,figsize=(22,6))
    fig.tight_layout(pad=3.0)
    sb.set_theme(style="whitegrid")
                                                           
    x_axis_to_plot = ["n_nodes","mean_size","mean_degree","density","n_nodes"]
    y_axis_to_plot = ["n_edges","n_nodes","n_nodes","n_edges","density"]
    axis_labels = {"n_nodes":"number of nodes",
                    "n_edges":"number of edges",
                    "mean_size":"nodes mean size",
                    "mean_degree":"nodes mean degree",
                    "density":"graph density"}
                                       
                                                   
    for n_plots in range(len(x_axis_to_plot)):
        scatterplot = sb.scatterplot(data = grid_search_results_df,x=x_axis_to_plot[n_plots],y=y_axis_to_plot[n_plots],hue='lens_name',s=120,ax=ax[n_plots],style="lens_name" ,legend=True)
        ax[n_plots].set_xticklabels([str(round(i,3)) for i in ax[n_plots].get_xticks()])
        plt.setp(scatterplot.get_legend().get_texts(), fontsize=16)
        plt.legend(title = "lens function name",title_fontsize=18,bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0)
        for lh in scatterplot.get_legend().legend_handles: 
            lh._sizes = [120] 
            
        # if annotate display the index of the row dataframe grid search on the scatterplot
        if (annotate):        
            for line in grid_search_results_df.index:
                if line == 30:
                    ax[n_plots].annotate(line,(grid_search_results_df[x_axis_to_plot[n_plots]][line], grid_search_results_df[y_axis_to_plot[n_plots]][line]), horizontalalignment='left', size='medium', color='black')
        ax[n_plots].set_xlabel(axis_labels[x_axis_to_plot[n_plots]],size=20)
        ax[n_plots].set_ylabel(axis_labels[y_axis_to_plot[n_plots]],size=20)
    plt.savefig("./results/mapper_statistics.png",dpi=300,bbox_inches='tight')

In [ ]:
# inspect the results
grid_search_results_optimal_choice("./results/lens_parameters_grid_search.xlsx",
                                   dataset_experiment['Y'],
                                   distance_matrix,
                                   n_dimension,
                                   random_seed,
                                   dataset_experiment[['X1','X2']],
                                   False)                                       

### 2.2) TDA Mapper pipeline - second step of the grid search

In [ ]:
# tune mapper hyparameters and different clustering algorithm 
grid_search_results_df = lenses_and_hyperparameters_greedy_search(distance_matrix,
                  ['tSNE'], n_dimension,
                  [40],[900], [300],
                  [],[],
                  np.array([14, 16, 18, 20, 22]),np.array([0.2, 0.3, 0.4, 0.5]), 
                  [DBSCAN(metric="precomputed",min_samples=2,eps=0.1,n_jobs=-1),
                   DBSCAN(metric="precomputed",min_samples=4,eps=0.1,n_jobs=-1),
                   DBSCAN(metric="precomputed",min_samples=2,eps=0.2,n_jobs=-1),
                   DBSCAN(metric="precomputed",min_samples=4,eps=0.2,n_jobs=-1),
                   DBSCAN(metric="precomputed",min_samples=2,eps=0.3,n_jobs=-1),
                   DBSCAN(metric="precomputed",min_samples=4,eps=0.3,n_jobs=-1),
                   DBSCAN(metric="precomputed",min_samples=2,eps=0.5,n_jobs=-1),
                   DBSCAN(metric="precomputed",min_samples=4,eps=0.5,n_jobs=-1),
                  AgglomerativeClustering(metric='precomputed', linkage='complete',n_clusters=2),
                  AgglomerativeClustering(metric='precomputed', linkage='complete',n_clusters=3),
                  SpectralClustering(n_clusters=2,assign_labels='kmeans', random_state=random_seed, affinity="precomputed",n_jobs=-1),
                  SpectralClustering(n_clusters=3,assign_labels='kmeans', random_state=random_seed, affinity="precomputed",n_jobs=-1)],
                  random_seed,
                  dataset_experiment['Y'], 
                  dataset_experiment[['X1','X2']], 
                  "./results/mapper_parameters_grid_search.xlsx",
                  True)


### Results

In [73]:
def evaluate_seach_Mapper_parameters(path_mapper_search_results_df,dataset_len):
    """
    Function to plot and evaluate the second step of the grid search through graph statistics.
    INPUT:
       - path_mapper_search_results_df: (string) path in which the second step grid search results is saved
       - dataset_len: (int) number of samples in the dataset
    OUTPUT:
       - df_plot: (pandas DataFrame) dataframe with the combinations of mapper parameters and the graph statistics.
    """
    mapper_search_results_df = read_grid_search_results(path_mapper_search_results_df)

    resolutios = []
    gains = []
    entropy = []
    mean_degre = []
    mean_size = []
    number_of_nodes = []
    zero_degree_boolean = []
    n_unique = []
    cluster_methods = []
    additional_paramss = []
    
    for index,row in mapper_search_results_df.iterrows():
        resolutios.append(int(row['parameters_combination'].split("n_cubes: ")[1][0:2]))
        gains.append(float(row['parameters_combination'].split("perc overlap: ")[1][0:3]))
        
        cluster_method = str(row['parameters_combination'].split("cluster method: ")[1]).split("(")[0]
        additional_params = ""
        if cluster_method == "DBSCAN":
            min_sample = str(str(row['parameters_combination'].split("cluster method: ")[1]).split("(")[1]).split("min_samples=")
            eps = str(str(row['parameters_combination'].split("cluster method: ")[1]).split("(")[1]).split("eps=")
            
            if len(eps)>1:
                additional_params += " eps = " + str(eps[1]).split(",")[0]
            else:
                additional_params += " eps = " + str(0.5)
                
            
            if len(min_sample)>1:
                additional_params += " min_sample = " + str(min_sample[1]).split(",")[0]  
            else:
                additional_params += " min_sample = " + str(2)
            
        else:
            n_clusters = str(str(row['parameters_combination'].split("cluster method: ")[1]).split("(")[1]).split("n_clusters=")[0][0]
            if len(n_clusters)>1:
                additional_params = " N = " + str(n_clusters) 
            else:
                additional_params = " N = " + str(2) 
            
        additional_paramss.append(additional_params)
        cluster_methods.append(cluster_method)
        entropy.append(row['graph_entropy'])
        mean_degre.append(row['mean_degree'])
        mean_size.append(row['mean_size'])
        number_of_nodes.append(row['n_nodes'])
        n_unique.append(row['n_unique'])
        
        # exclude the combinations that leads to a graph with isolated nodes
        if 0 in row['nodes_degree']:
            zero_degree_boolean.append(True)
        else:
            zero_degree_boolean.append(False)

    df_plot = pd.DataFrame({"Resolution":resolutios, "Gain":gains, "n_nodes":number_of_nodes,
                           "mean degree":mean_degre, "mean size":mean_size,
                           "zero_degree_boolean":zero_degree_boolean,"n_unique":n_unique, 
                            "cluster_methods":cluster_methods, "additional_params":additional_paramss})
    df_plot = df_plot[(df_plot['zero_degree_boolean']==False) & (df_plot['n_unique']==dataset_len)]

    # plot graph statistics and optionally incudes legend by colouring the markers according to the clustering method
    fig, ax = plt.subplots(3,2,figsize=(8,9),sharex="col",sharey="row")
    fig.tight_layout(pad=2.0)

    sb.scatterplot(data = df_plot, x = "Resolution", y = "n_nodes",hue = "cluster_methods",style = "cluster_methods",s =110 ,ax=ax[0,0],palette = "tab10",legend=False)
    sb.scatterplot(data = df_plot, x = "Gain", y = "n_nodes",hue = "cluster_methods",style = "cluster_methods",s = 110,ax=ax[0,1],palette = "tab10",legend=False)
    
    sb.scatterplot(data = df_plot, x = "Resolution", y = "mean degree",hue = "cluster_methods",style = "cluster_methods",s =110,ax=ax[1,0],palette = "tab10",legend=False)
    sb.scatterplot(data = df_plot, x = "Gain", y = "mean degree",hue = "cluster_methods",style = "cluster_methods",s =110 ,ax=ax[1,1],palette = "tab10",legend=False)
    
    sb.scatterplot(data = df_plot, x = "Resolution", y = "mean size",hue = "cluster_methods",style = "cluster_methods",s = 110,ax=ax[2,0],palette = "tab10",legend=False)
    scatter = sb.scatterplot(data = df_plot, x = "Gain", y = "mean size",hue = "cluster_methods",style = "cluster_methods",s = 110,ax=ax[2,1],palette = "tab10",legend=True)
    
    #plt.setp(scatter.get_legend().get_texts(), fontsize=18)
    #plt.legend(title = "Clustering method",bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0,title_fontsize=16)
#     for lh in scatter.get_legend().legendHandles: 
#             lh._sizes = [150]       
        
    ax[2,0].set_xlabel("resolution",size=18)
    ax[2,1].set_xlabel("gain",size=18)
    ax[0,0].set_ylabel("Number of nodes",size=18)
    ax[1,0].set_ylabel("Nodes mean degree",size=18)
    ax[2,0].set_ylabel("Nodes mean size",size=18)
    ax[2,0].tick_params(axis='both', labelsize=14)
    ax[2,1].tick_params(axis='both', labelsize=14)
    ax[0,0].tick_params(axis='both', labelsize=14)
    ax[1,0].tick_params(axis='both', labelsize=14)
    ax[2,0].tick_params(axis='both', labelsize=14)
    
    plt.savefig("./results/mapper_params.png",dpi=300,bbox_inches='tight')
    return df_plot

In [ ]:
# inspect the results
second_step_df_plot = evaluate_seach_Mapper_parameters('./results/mapper_parameters_grid_search.xlsx',dataset_experiment.shape[0])

### 2.3) TDA Mapper pipeline - apply the algorithm

In [104]:
def set_node_community(G, communities):
    """
    Function that assing to each node in the networkx graph the community as attribute
    INPUT:
    - G:           (networkx graph)  networkx graph obtained from the Mapper simplicial complex
    - communities: (list of int)    list of integers, containing the community assigned to each node in the graph G
    """
    for c, nodes_community_c in enumerate(communities):
        for node_c in nodes_community_c:
            G.nodes[node_c]['community'] = c + 1
     
def set_edge_community(G):
    """
    Function which searches for edges within the community and adds them.
    INPUT:
    - G:           (networkx graph)  networkx graph obtained from the Mapper simplicial complex        
    """
    for v, w, in G.edges:
        if G.nodes[v]['community'] == G.nodes[w]['community']:
            # Internal edge marked with the community (number)
            G.edges[v, w]['community'] = G.nodes[v]['community']
        else:
            # External edge marked with a 0
            G.edges[v, w]['community'] = 0
     
def get_color(i, r_off=1, g_off=1, b_off=1):
    """
    Function that assign the same color to the nodes in a community.
    INPUT:
       -i: (int) integer that define the community id
    OUTPUT:
       - (r, g, b): tuple indicating the community's colour, containing the levels of red, green and blue.
    """
        
    r0, g0, b0 = 0, 0, 0
    n = 16
    low, high = 0.1, 0.9
    span = high - low
    r = low + span * (((i + r_off) * 3) % n) / (n - 1)
    g = low + span * (((i + g_off) * 5) % n) / (n - 1)
    b = low + span * (((i + b_off) * 7) % n) / (n - 1)
    return (r, g, b)



def associate_patients_to_communities(G,scomplex,dataset,strategy):
    """
    Function that assing to each patient a node community (assign a subgroup).
    INPUT:
       - G:           (networkx graph)  networkx graph obtained from the Mapper simplicial complex   
       - scomplex:    (dictionary) simplicial complex resulting from the application of KeplerMapper
       - dataset     (pandas DataFrame) patients dataset
       - strategy    (string) user defined strategy to resolve ties
    OUTPUT:
       - new_dataset   (pandas DataFrame) patients dataset with an additional column, indicating the novel subroup.
    """       


    # compute the laplacian centrality for each node if this is the strategy
    if strategy == "laplacian centrality":
        dict_strategy = {}
        for v, c in nx.centrality.laplacian_centrality(G,weight="weight").items():
            dict_strategy[v] = c

    # compute the pagerank centrality for each node if this is the strategy
    if strategy == "pagerank centrality":
        dict_strategy = nx.pagerank(G,weight="weight")
    
    # loop over patiets ids and graph nodes, to find all the nodes in which the patients appears
    assigned_communities = []
    for j in dataset.iterrows():
        patient_nodes = {}
        for key,values in scomplex["nodes"].items():
            if j[0] in values:
                patient_nodes[key] = G.nodes[key]['community']
                
                
        # Compute lists that contain:
        
        # all the communities in which a patient appears 
        candidate_communities = list(pd.Series(patient_nodes.values(),dtype=int).value_counts().index )     
        # and its frequency (how many nodes FROM A SPECIFIC COMMUNITY contain that patient? )
        canditate_frequencies = list(pd.Series(patient_nodes.values(),dtype=int).value_counts().values) 
        # note that this list are ordered according to the frequencies
        
        # case 1: the patient appears only in one node -> directly assing the community
        if len(candidate_communities)==1:
            assigned_communities.append(candidate_communities[0])
            
        # case 2: the patient appears in more than one node
        elif len(candidate_communities)>1:

            # if there is a community of majority (there is a community in which the patient appear frequently )
            if canditate_frequencies[0]>canditate_frequencies[1]:
                assigned_communities.append(candidate_communities[0])

            # otherwise resolve ties according to the user defined strategy
            else:

                max_frequency = max(canditate_frequencies)
                index_max = [i for i, j in enumerate(canditate_frequencies) if j == max_frequency]

                list_community_number_of_patients = []
                        
                for index in index_max:
                    total_n_patients_in_all_nodes_for_specific_community = 0
                    for node,community in patient_nodes.items():
                        if community == candidate_communities[index]:

                            # assign according to the node size
                            if strategy == "node size":
                                total_n_patients_in_all_nodes_for_specific_community += len(scomplex['nodes'][node])
                            elif strategy == "node degree":
                                total_n_patients_in_all_nodes_for_specific_community += G.degree([node],weight="weight")[node]
                            elif strategy == "laplacian centrality" or strategy == "pagerank centrality":
                                total_n_patients_in_all_nodes_for_specific_community += dict_strategy[node]

                    list_community_number_of_patients.append(total_n_patients_in_all_nodes_for_specific_community)

                assigned_communities.append(candidate_communities[list_community_number_of_patients.index(max(list_community_number_of_patients))])  
               
        # a patient  not assigned to any community is marked with the community 0
        else:
            assigned_communities.append(0)
 
    new_dataset = dataset.copy(deep=True)
    new_dataset['communities'] = assigned_communities   
    return new_dataset

def enrich_topology(scomplex,enrichment_feature, colormap, axs_to_plot, colours_method, communities_separated,
                    internal,internal_color,colorbar_labelsize,colorbar_ticks_size):
    
    """
    Function that allows to enrich (i.e. to color) the simplicial complex with a variable.
    INPUT:
       - scomplex:    (dictionary) simplicial complex resulting from the application of KeplerMapper
       - enrichment_feature: (pandas Series) dataset's column indicating the enrichment feature
       - colormap: (string) matplotlib colormap to use to enrich the simplicial complex
       - axs_to_plot: (matplotlib.axes) axes in which plot the enrichment
       - colours_method: (string) string that specify the type of the feature for the enrichment
       - communities_separated: (boolean) flag that indicates if in the plot the communities are drawn separately
       - internal: (list) list of edges belonging to communities
       - internal_color: (list) different communities colours
       - colorbar_labelsize: (int) size of the colorbar labels
       - colorbar_ticks_size: (int) size of the colorbar tick
    """
    kmgraph,  mapper_summary, colorf_distribution = get_mapper_graph(scomplex)
    G = km.adapter.to_nx(scomplex)
    new_color = []


    if colours_method == "categorical":

        # Build a mapping of all unique classes
        unique_classes = sorted(enrichment_feature.unique())
        dict_categorical = {cls: f"Class_{i}" for i, cls in enumerate(unique_classes)}
        class_to_index = {cls: i for i, cls in enumerate(unique_classes)}  # For coloring

        for j, node in enumerate(kmgraph['nodes']):
            # Get the member labels for this node
            member_label_ids = enrichment_feature[scomplex['nodes'][node['name']]]
            member_labels = [dict_categorical[id] for id in member_label_ids]

            # Count how many of each class
            label_counts = Counter(member_labels)
            # Find the most common class
            most_common_label, _ = label_counts.most_common(1)[0]

            # Find the corresponding class index for color assignment
            original_class_value = [k for k, v in dict_categorical.items() if v == most_common_label][0]
            color_value = class_to_index[original_class_value] * 1.0  # convert to float if needed
            
            new_color.append(color_value)

    # if the enrichment variable is numerical -> colors according to the mean of the variables
    elif colours_method == "numerical":
        for j, node in enumerate(kmgraph['nodes']):
            member_feature = enrichment_feature[scomplex['nodes'][node['name']]]       
            new_color.append(np.mean(member_feature))
        
    node_sizes = [len(scomplex["nodes"][node]) * 10 for node in G.nodes()]

    # if communities_separated is True, plot the graph with communities separated
    if communities_separated:
        nx.draw_kamada_kawai(G,node_color = new_color, edgelist=internal, edge_color = internal_color, node_size=60,cmap = colormap,ax=axs_to_plot)
    else:
        nx.draw_kamada_kawai(G,node_color = new_color, node_size=60,cmap = colormap,ax=axs_to_plot)
       
    # associate to the graph plot a colorbar
    sm = plt.cm.ScalarMappable(cmap=colormap, norm=plt.Normalize(vmin=0, vmax=1))
    sm.set_array([])
    cbar = plt.colorbar(sm,ax=axs_to_plot, pad=0,fraction=0.05)
    cbar.set_label(label=enrichment_feature.name,size=colorbar_labelsize) 
    #cbar.set_ticks([0.2, 0.5, 0.8])  # Set tick positions
    #cbar.set_ticklabels([])  # Set tick labels
    cbar.ax.tick_params(labelsize=colorbar_ticks_size)

def community_searching(G, method, random_seed):
    """
    Function which allows to apply the communities discovering algorithm.
    INPUT:
    - G:           (networkx graph)  networkx graph obtained from the Mapper simplicial complex   
    - method:      (string)  string specifying the algorithm to use
    - random_seed           (int) random seed for reproducibility
    OUTPUT:
    - node_color: (list) list of tuples containing the nodes colours in rgb 
    - internal: (list) list of edges belonging to communities
    - internal_color: (list) different communities colours
    - external: (list) list of edges external to communities 
    - [coverage,perfomance,modularity]:  (list) list of the partition perfomance scores
    """
    
    # apply the communities discovering algorithm
    if method == "Greedy modularity":
        communities = nxcom.greedy_modularity_communities(G, weight = 'weight')
    elif method == "Louvain":
        communities = nxcom.louvain_communities(G, weight = 'weight', seed = random_seed)
    elif method == "Girvan Newman":
        comm = nxcom.girvan_newman(G)
        communities = [sorted(c) for c in next(comm)]

    # evaluate the partition with scores performance
    coverage, perfomance = nxcom.partition_quality(G,communities)
    modularity = nxcom.modularity(G,communities, weight = 'weight')       
        
    # Define the nodes and the edges communities
    set_node_community(G, communities)
    set_edge_community(G)
        
    # get the colors of each node
    node_color = [get_color(G.nodes[v]['community']) for v in G.nodes]

    # Importing the colour of edges between members of the same community (internal) 
    # and edges between different communities
    external = [(v, w) for v, w in G.edges if G.edges[v, w]['community'] == 0]
    internal = [(v, w) for v, w in G.edges if G.edges[v, w]['community'] > 0]
    internal_color = ['black' for e in internal]
    return node_color, internal, internal_color,external,[coverage,perfomance,modularity]
    

def TDA_patiets_phenotyping_pipeline(distance_matrix,projection_lens,resolution,p_overlap,cluster_method, 
                                     weighted, continue_feature,
                                     categorical_feature, categorical_feature_colormap, 
                                     community_detection_algorithm,
                                     dataset,plots, flag_remove_duplicate_nodes, random_seed, id_paz):
    '''
        Function that wrap the overall TDA pipeline.
        INPUT:
        - distance_matrix:      (numpy ndarray) patients distance matrix
        - projection_lens:      (sklearn or umap-learn object) the projection lens to use
        - resolution :          (int) resolution parameter to use
        - p_overlap :           (float) gain parameter to use
        - cluster_method:       (sklearn cluster method) cluster method to use
        - weighted:             (bool) if True, specify that the graph obtained with KeplerMapper will be weighted
        - continue_feature :    (string) continue features, used to weight the graph edges
        - categorical_feature:  (string) categorical feature, used to enrich the graph and for sankey diagram
        - categorical_feature_colormap: (string) matplotlib colormap to use to enrich the simplicial complex
        - community_detection_algorithm: (string) communities detection algorithm to use
        - dataset :              (pandas DataFrame) patients dataset
        - plots:                (boolean) flag used to specify that the results will be plotted
        - flag_remove_duplicate_nodes    (boolean) if remove duplicated node from the scomplex in output
        - random_seed           (int) random seed for reproducibility
        - id_paz                (string) string specifying the name of the column id in the dataset
        OUTPUT:
        - dataset_with_communities   (pandas DataFrame) patients dataset with an additional column, indicating the novel subroup.
        - scomplex:    (dictionary) simplicial complex resulting from the application of KeplerMapper
        - (internal, internal_color) tuples of list of edges belonging to communities and different communities colours
        - node_color: (list) list of tuples containing the nodes colours in rgb 
        - G:           (networkx graph)  networkx graph obtained from the Mapper simplicial complex   
    '''

    mapper = km.KeplerMapper()  
    # apply the KeplerMapper pipeline to obtain a simplicial complex
    projection = mapper.project(distance_matrix, projection= projection_lens,distance_matrix = None, scaler = None)
    scomplex = mapper.map(projection, distance_matrix, 
                cover=km.Cover(n_cubes=resolution, perc_overlap=p_overlap), clusterer = cluster_method,  precomputed=True,
                                     remove_duplicate_nodes = True)
    
    # from simplicial complex to networkx graph
    G = km.adapter.to_nx(scomplex)
    
    # weighted string it not empy -> add weights to the graph edges
    if weighted!="":
        for edge in G.edges:
            node_A = scomplex['nodes'][edge[0]]
            node_B = scomplex['nodes'][edge[1]]
            
            # weight the edges with the number of patients in common between the nodes
            if weighted == "intersection_size":
                G[edge[0]][edge[1]]['weight']  = len(set(node_A).intersection(set(node_B)))
    
    # search communities
    node_color, internal, internal_color, external,scores = community_searching(G,community_detection_algorithm, random_seed)
    

    # visualization
    if plots!=False:
    
        f, axs = plt.subplots(1,2,figsize=(16,6),layout="tight")
        
        # the graph created with KeplerMapper
        nx.draw_kamada_kawai(G, node_size=90,ax=axs[0])
        
        # the graph enriched with the initial phenotype and communities separated
        enrich_topology(scomplex,dataset[categorical_feature],categorical_feature_colormap,axs[1],"categorical",True,internal,internal_color,28,20)

        plt.savefig('./results/tda_phenotyping_output',bbox_inches="tight",dpi=400)
        
    # associate patients to communities
    dataset_with_communities = associate_patients_to_communities(G,scomplex,dataset,"node size")
    dataset_with_communities['Patient_ID'] = id_paz
    
    dataset_with_communities.to_excel("./results/cd_dataset_with_comunities.xlsx",index=False)

    return dataset_with_communities,scomplex,(internal, internal_color),node_color, G

In [ ]:
# define the hyperparameters found before
projection_lens = TSNE(n_components=n_dimension, 
                      random_state=random_seed, 
                      perplexity = 40 , 
                      learning_rate= 900 , 
                      max_iter= 300,
                      init="random",
                      metric="precomputed")

mapper_resolution = 14
mapper_gain = 0.5
mapper_cluster_method = DBSCAN(metric="precomputed",min_samples=2,eps=0.3,n_jobs=-1)
community_detection_algorithm = ""

# plotting params
phe_colors = [    (0.0,'green'),   (0.5,'yellow'),    (1.0,'orange') ]
colormap = LinearSegmentedColormap.from_list("qualitative_set", phe_colors, N=3)

# run the algorithm and inspect the results
dataset_with_communities,scomplex,tuple_internal_color,node_color, G = TDA_patiets_phenotyping_pipeline(distance_matrix, 
                                                                                                        projection_lens,
                                                                                                        mapper_resolution, 
                                                                                                        mapper_gain , 
                                                                                                        mapper_cluster_method, 
                                                                                                        True, 
                                                                                                        continue_features,
                                                                                                        'Y', 
                                                                                                        colormap, 
                                                                                                        'Louvain',
                                                                                                        dataset_experiment.loc[:, ~dataset_experiment.columns.isin(['Patient_ID'])],
                                                                                                        True, 
                                                                                                        True,
                                                                                                        random_seed, 
                                                                                                        dataset_experiment['Patient_ID'])




#### ---------------------------------------------------------------------------------------------------------------------------------------------------------

### 3) Predictive modeling

In [125]:
def make_dummies_and_scale(dataset, patient_id, categorical_features, binary_features, continue_features, encode_flag, scale_flag):
    """
    Function that makes dummies variables from categorical variables and standardize the numerical variables.
    INPUT:
        - dataset :              (pandas DataFrame) patients dataset
        - patient_id:            (pandas Series) dataset's column indicating the samples IDs
        - categorical_features:  (list) list of dataset multicategorical features
        - binary_features:       (lsit) list of datset binary features
        - continue_features:     (list) list of dataset numerical features
        - encode_flag:           (boolean) flag used to specify that we want to make one-hot variables from categorical ones (n-1 level for binary, n levels for multicategorical)
        - scale_flag:            (boolean) flag used to specify that we want to standardize numerical variables.
    OUTPUT:
        - dataset:               (pandas DataFrame) patients dataset with encoded and standardized variables 
    """


    dataset_copy = dataset.copy(True)

    # encode categorical with one-hot, while binary variables are keeped as is
    if encode_flag:

        if (len(categorical_features)!=0) and (len(binary_features)==0):

            df_categorical_dummies = pd.get_dummies(dataset_copy[categorical_features].astype(str),drop_first=False)
            df_others = dataset_copy.loc[:,~dataset_copy.columns.isin(categorical_features)]

            dataset_copy = pd.concat([df_others,df_categorical_dummies],axis=1)

        elif (len(categorical_features)==0) and (len(binary_features)!=0):

            df_binary_dummies = pd.get_dummies(dataset_copy[binary_features].astype(str),drop_first=True)
            df_others = dataset_copy.loc[:,~dataset_copy.columns.isin(binary_features)]

            dataset_copy = pd.concat([df_others,df_binary_dummies],axis=1)

        elif (len(categorical_features)!=0) and (len(binary_features)!=0):
            
            df_categorical_dummies = pd.get_dummies(dataset_copy[categorical_features].astype(str),drop_first=False)
            df_binary_dummies = pd.get_dummies(dataset_copy[binary_features].astype(str),drop_first=True)
            df_others = dataset_copy.loc[:,~dataset_copy.columns.isin(categorical_features + binary_features)]

            dataset_copy = pd.concat([df_others,df_categorical_dummies,df_binary_dummies],axis=1)

    # scale numerical
    if scale_flag:

        scaler = preprocessing.StandardScaler()
        df_numerical = dataset_copy.loc[:, dataset_copy.columns.isin(continue_features)]
        df_others = dataset_copy.loc[:,~dataset_copy.columns.isin(continue_features)]
        X_numerical_scaled = scaler.fit_transform(df_numerical)
        df_train_numerical_scaled = pd.DataFrame(X_numerical_scaled,columns=df_numerical.columns)
        dataset_copy = pd.concat([df_others,df_train_numerical_scaled],axis=1)

    return dataset_copy

def training_classifier(X_train, y_train,classifier, rnd,categorical_features,binary_features,continue_features,id_paz,cv_split,community):
    """
    Function that train a classifier model with cross validation, while tuning its parameters with a grid search.
    Return the variables selected for each model and the model with the best hyperparameters fitted on X_train.
    INPUT:
        - X_train: (pandas DataFrame) patients features 
        - y_train: (pandas Series) binary class 
        - classifier: (string) name of the classifier
        - rnd: (int) seed for reproducible output
        - categorical_features: (list) list of dataset multicategorical features
        - binary_features:  (list) binary features of the dataset
        - continue_features: (list) list of dataset numerical features
        - id_paz: (pandas Series) dataset's column indicating the samples IDs
        - cv_split: (int) number of fold K for K-cross validation
        - community: (int) community  id
    OUTPUT:
        - variables_selected: (dict) dictionary with variables:variables_importance, selected by the model.
        - grid_search: (sklearn GridSearchCV) fitted model with the optimal combination of hyperparameters, fitted on X_train.
    """
        
    # define the metrics for the hyperparameters grid search
    score = ["roc_auc"]
        
    if classifier == "logistic regression":
        X_train = make_dummies_and_scale(X_train,id_paz,categorical_features,binary_features, continue_features, True, True)
        log_reg = LogisticRegression(penalty="elasticnet",solver="saga",random_state=rnd)
        parameters_grid = {"C" : [1000, 100, 10, 1, 0.1,0.01,0.001],"l1_ratio" : [0.25,0.5,0.75]}
            
        grid_search = GridSearchCV(estimator = log_reg, param_grid = parameters_grid, 
                            cv = 5, n_jobs = -1,scoring=score,refit = score[0])
        grid_search.fit(X_train, y_train)
        coef_value = grid_search.best_estimator_.coef_
        coef_name = X_train.columns
        
    if classifier == "random forest":
        X_train = make_dummies_and_scale(X_train,id_paz,categorical_features,binary_features, continue_features, True, False)
        rf = RandomForestClassifier(max_features = "sqrt", random_state = rnd,bootstrap = False)
        parameters_grid = {'n_estimators': [100,200,300],
                        'max_depth': [1,3,5],
                        'min_samples_split': [2, 5, 10],
                        'min_samples_leaf': [1, 5]}

        grid_search = GridSearchCV(estimator = rf, param_grid = parameters_grid, 
                            cv = 5, n_jobs = -1,scoring=score,refit = score[0])
        grid_search.fit(X_train, y_train)
        
        coef_value = grid_search.best_estimator_.feature_importances_
        coef_name = grid_search.best_estimator_.feature_names_in_  

    elif classifier == "XGBoost":
        X_train = make_dummies_and_scale(X_train,id_paz,categorical_features, binary_features,continue_features, True, False)
        xgb_model = xgb.XGBClassifier(objective="binary:logistic", random_state=rnd, booster="gbtree")
        parameters_grid = {"learning_rate":[0.1, 0.25, 0.5], "gamma":[0, 0.1, 0.2, 0.3],
                            'max_depth': [1,3,5], "n_estimators":[100,200,300] }

        grid_search = GridSearchCV(estimator = xgb_model, param_grid = parameters_grid, 
                            cv = 5, n_jobs = -1,scoring=score,refit = score[0])
        grid_search.fit(X_train, y_train)
        
        coef_value = grid_search.best_estimator_.feature_importances_
        coef_name = grid_search.best_estimator_.feature_names_in_

    print("Community " + str(community) + "- Best model parameters: " + str(grid_search.best_params_) + " that leads to the following score: "+ str(grid_search.best_score_))
    
    # compute classification score, ROC curves and PR curves
    compute_classification_score(X_train, y_train , grid_search,classifier, community)
    
    if classifier == "logistic regression":
        # extract features importance 
        coef = pd.Series(coef_value[0],index=coef_name)
    else:
        coef = pd.Series(coef_value,index=coef_name)

    
    #take the variables with importance more than 0 and sort their values 
    coef = coef[coef>0]  
    if len(coef)>5:
        coef = coef[:5]

    coef = coef.sort_values(ascending=False)
    variables_selected = coef.to_dict()

    joblib.dump(variables_selected, "Features community " + str(community) + " - classifier " + str(classifier) + ".pkl")
    
    return variables_selected,grid_search
    
def plot_variable_importance(classifier,coef_enrichment,scomplex,tuple_internal_color,dataset,continue_features,community):
    """
    Function that enrich the simplicial complex with the features that have a variable importance greather than 0
    """
    if len(imp_coef)>5:
        coef_enrichment = imp_coef[:5]
    else:
        coef_enrichment = imp_coef

    n_var = len(coef_enrichment)
    n_col = 2
    n_row = math.ceil(n_var/n_col)

    if n_row == 1:

        f, axs = plt.subplots(n_row,n_col,figsize=(14,n_row*5),layout="tight")
        c = 0
            
        for key,value in coef_enrichment.to_dict().items():
                
            if c == 2:
                c = 0
                
            # if the variable is categorical
            if key not in continue_features:
                enrich_topology(scomplex,dataset[key],'Blues',axs[c],"categorical",True,tuple_internal_color[0],tuple_internal_color[1],28,20)
            # if the variable is numerical
            else:
                enrich_topology(scomplex,dataset[key], 'Oranges', axs[c], "numerical",True,tuple_internal_color[0],tuple_internal_color[1],28,20 )
                
            c+=1

        plt.savefig("./results/enriched_topology_" + classifier + "_" + str(community),bbox_inches="tight",dpi=400)

    elif n_row > 1:

        f, axs = plt.subplots(n_row,n_col,figsize=(14,n_row*5),layout="tight")
        c = r = 0
            
        for key,value in coef_enrichment.to_dict().items():
                
            if c == 2:
                c = 0
                r +=1
                
            # if the variable is categorical
            if key not in continue_features:
                enrich_topology(scomplex,dataset[key],'Blues',axs[r,c],"categorical",True,tuple_internal_color[0],tuple_internal_color[1],28,20)
            # if the variable is numerical
            else:
                enrich_topology(scomplex,dataset[key], 'Oranges', axs[r,c], "numerical",True,tuple_internal_color[0],tuple_internal_color[1],28,20 )
                
            c+=1

        plt.savefig("./results/enriched_topology_" + classifier + "_" + str(community),bbox_inches="tight",dpi=400)

    else:
        pass

def plot_variable_distribution(classifier,coef_enrichment,scomplex,tuple_internal_color,dataset,continue_features,features_imp):
    """
    Function that enrich the simplicial complex with the features that have a variable importance greather than 0
    """
    if len(imp_coef)>5:
        coef_enrichment = imp_coef[:5]
    else:
        coef_enrichment = imp_coef
    features_to_plot = {}

    for key,value in coef_enrichment.to_dict().items(): 

        if key in features_imp:
            features_to_plot[key]=value

    custom_colors = [
        'gainsboro', # first color
        'black' # last color
        ]

    custom_cmap = LinearSegmentedColormap.from_list("custom_gradient", custom_colors)
                
    for key,value in features_to_plot.items():

        plt.figure()
        
        # if the variable is categorical
        if key not in continue_features:
            enrich_topology_singleplot(scomplex,dataset[key],custom_cmap,"categorical",True,tuple_internal_color[0],tuple_internal_color[1],28,20)
        # if the variable is numerical
        else:
            enrich_topology_singleplot(scomplex,dataset[key], 'Oranges', "numerical",True,tuple_internal_color[0],tuple_internal_color[1],28,20 )
                

        plt.savefig("./results/enriched_topology_" + classifier +  "_feature_" + key, bbox_inches="tight",dpi=400)


def compute_classification_score(X_train,y_true,fitted_model,classifier, community_to_classify):
    """
    Function that computes different classification score, ROC and PR curves.
    INPUT:
        - X_train: (pandas DataFrame) patients features 
        - y_true:  (pandas Series) outcome to predict
        - fitted_model: (sklearn GridSearchCV) fitted model with the optimal combination of hyperparameters, fitted on X_train
        - classifier: (string) classifier name
    """
    prob_pred = fitted_model.predict_proba(X_train)
    label_pred = fitted_model.predict(X_train)
    
    mcc = matthews_corrcoef(y_true,label_pred)
    f1 = f1_score(y_true,label_pred)
    precision = precision_score(y_true,label_pred)
    sentivity = recall = recall_score(y_true,label_pred)
    cm = confusion_matrix(y_true,label_pred)
    specificity = cm[0,0]/(cm[0,1]+cm[0,0])
    PPV = cm[1,1]/(cm[0,1]+cm[1,1])
    NPV = cm[0,0]/(cm[1,0]+cm[0,0])
    
    auc_roc = roc_auc_score(y_true,prob_pred[:,1])
    brier = brier_score_loss(y_true,prob_pred[:,1])
    
    fpr, tpr, thresholdsROC = roc_curve(y_true, prob_pred[:,1], pos_label=1)
    precisionr, recallr, thresholdsPR = precision_recall_curve(y_true, prob_pred[:,1], pos_label=1)
    auc_pr = auc(recallr, precisionr)
    
    f = plt.figure(figsize=(8,8),layout="tight")
    gs = plt.GridSpec(2, 2, figure=f)
    ax1 = f.add_subplot(gs[0, 0])
    ax2 = f.add_subplot(gs[1, 0])
    ax3 = f.add_subplot(gs[:, 1])
    ax1.grid(True)
    ax1.plot(fpr,tpr)
    ax1.set_xlabel("False positive rate (1-specificity)",fontsize=14)
    ax1.set_ylabel("True positive rate (sensitivity)",fontsize=14)
    ax1.set_xticklabels([str(round(i,3)) for i in ax1.get_xticks()], fontsize = 13)
    ax1.set_yticklabels([str(round(i,3)) for i in ax1.get_yticks()], fontsize = 13)
    ax1.set_title(label="ROC curve",fontsize=14)
    ax2.grid(True)
    ax2.plot(precisionr,recallr)
    ax2.set_xticklabels([str(round(i,3)) for i in ax2.get_xticks()], fontsize = 13)
    ax2.set_yticklabels([str(round(i,3)) for i in ax2.get_yticks()], fontsize = 13)
    ax2.set_xlabel("Recall",fontsize=14)
    ax2.set_ylabel("Precision",fontsize=14)
    ax2.set_title(label="PR curve",fontsize=14)
    
    
    list_table_print = [["AUC - ROC: " + str(round(auc_roc,4))],
                        ["sentivity: " + str(round(sentivity,4))],
                        ["specificity: " + str(round(specificity,4))],
                        ["PPV: " + str(round(PPV,4))],
                        ["NPV: " + str(round(NPV,4))],
                        ["AUC - PR: " + str(round(auc_pr,4))],
                        ["precision: " + str(round(precision,4))],
                        ["recall: " + str(round(recall,4))],
                        ["f1: " + str(round(f1,4))],
                        ["MCC: " + str(round(mcc,4))],
                        ["brier: " + str(round(brier,4))]]
    columns = ("Classifier: ", classifier)
    ax3.axis('tight')
    ax3.axis('off')
    the_table = ax3.table(cellText=list_table_print, colLabels=columns, loc='center',colWidths=[0.5,0.5])
    
    plt.savefig("./results/classification_results_" + classifier + "_" + str(community_to_classify),bbox_inches="tight",dpi=400)

def computational_phenotyping(dataset_with_communities,features_to_exclude,classifier,rnd,categorical_features,binary_features,
                              continue_features,id_paz,cv_split,scomplex,tuple_internal_color,
                             variable_imp, graph_networkx):
    """
    Function that wrap the computational phenotyping.
    INPUT:
        - dataset_with_communities   (pandas DataFrame) patients dataset with an additional column, indicating the novel subroup.
        - features_to_exclude: (list) a list of columns to exclude from the dataset (in order to obtain only clinical features)
        - classifier: (string) classifier name
        - rnd         (int) random seed for reproducibility
        - categorical_features  (list) multicategorical features in the dataset
        - binary_features       (list) binary features in the dataset
        - continue_features     (list) numerical features in the dataset
        - id_paz: (pandas Series) dataset's column indicating the samples IDs
        - cv_split: (int) number of fold K for K-cross validation
        - scomplex:    (dictionary) simplicial complex resulting from the application of KeplerMapper
        - tuple_internal_color:  tuples of list of edges belonging to communities and different communities colours
        - variable_imp: (boolean) flag for visualizing the enrichment graph for each variables
        - graph_networkx : simplicial complex as networkx graph
    OUTPUT:
        - variables_selected: (dictionary) nested dictionary -> community:dict_var , with dict_var -> variable:variable_importance
        - best_model_community: (dictionary) dictionary of  community:best_fitted_model
    """
    
    communities = sorted(set(dataset_with_communities['communities']))

    print('The communities are ' + str(communities))

    variables_selected = {}
    best_model_community = {}
    
    X_train = dataset_with_communities.loc[:, ~dataset_with_communities.columns.isin(features_to_exclude)].copy(True)
    
    # for each community define the y_class (the community itself)
    for community in communities:
        y_class = dataset_with_communities['communities'] == community 
        y_class = y_class.replace(to_replace=False,value=0)
        y_class = y_class.replace(to_replace=True,value=1)

        
        #train a classifier and obtain variables importance
        variables_selected_community,fitted_model = training_classifier(X_train, y_class.astype(int),classifier, rnd,categorical_features,binary_features,continue_features,id_paz,cv_split,community)
        variables_selected[community] = variables_selected_community
        best_model_community[community] = fitted_model

        print('Analysis on community ' + str(community) + ' DONE')

    
    # make dummies in order to plot the enrichment
    dataset_with_dummies = make_dummies_and_scale(X_train,id_paz,categorical_features, binary_features, continue_features, True, None)

    if variable_imp:
        sb.set_style("whitegrid")
        for community,variables in variables_selected.items():
            list_color_edge_communities = []
            color_edge_communities = []

            #for i in features_to_plot:
             #   if var

            # define the edge color to highlight the community in the plot
            for index,edge in enumerate(tuple_internal_color[0]):
                if graph_networkx.edges[edge]['community'] == community:
                    list_color_edge_communities.append("black")
                else:
                    list_color_edge_communities.append("gainsboro")
                
               

            tuple_internal_color = (tuple_internal_color[0],list_color_edge_communities)

            x = pd.Series(variables)

            dataset_enrich = dataset_with_dummies
            plot_variable_importance(classifier,pd.Series(variables),scomplex,tuple_internal_color,dataset_enrich,continue_features,community)
        

    
    return variables_selected,best_model_community, x 


def computational_phenotyping_singleplot(dataset_with_communities,features_to_exclude,classifier,rnd,categorical_features,binary_features,
                              continue_features,id_paz,cv_split,scomplex,tuple_internal_color,
                             variable_imp, graph_networkx,features_imp,variables_selected):
    """
    Function that wrap the computational phenotyping.
    INPUT:
        - dataset_with_communities   (pandas DataFrame) patients dataset with an additional column, indicating the novel subroup.
        - features_to_exclude: (list) a list of columns to exclude from the dataset (in order to obtain only clinical features)
        - classifier: (string) classifier name
        - rnd         (int) random seed for reproducibility
        - categorical_features  (list) multicategorical features in the dataset
        - binary_features       (list) binary features in the dataset
        - continue_features     (list) numerical features in the dataset
        - id_paz: (pandas Series) dataset's column indicating the samples IDs
        - cv_split: (int) number of fold K for K-cross validation
        - scomplex:    (dictionary) simplicial complex resulting from the application of KeplerMapper
        - tuple_internal_color:  tuples of list of edges belonging to communities and different communities colours
        - variable_imp: (boolean) flag for visualizing the enrichment graph for each variables
        - graph_networkx : simplicial complex as networkx graph
    OUTPUT:
        - variables_selected: (dictionary) nested dictionary -> community:dict_var , with dict_var -> variable:variable_importance
        - best_model_community: (dictionary) dictionary of  community:best_fitted_model
    """
    
    X_train = dataset_with_communities.loc[:, ~dataset_with_communities.columns.isin(features_to_exclude)].copy(True)
    
    # make dummies in order to plot the enrichment
    dataset_with_dummies = make_dummies_and_scale(X_train,id_paz,categorical_features, binary_features, continue_features, True, None)

   
    sb.set_style("whitegrid")
    #variables = variables_selected.values()

    for community,variables in variables_selected.items():
        
        color_edge_communities = []

        # define the edge color to highlight the community in the plot
        for index,edge in enumerate(tuple_internal_color[0]):
            
            if graph_networkx.edges[edge]['community'] == 1:
                color_edge_communities.append("blue")
            elif graph_networkx.edges[edge]['community'] == 2:
                color_edge_communities.append("orange")
            elif graph_networkx.edges[edge]['community'] == 3:
                color_edge_communities.append("green")
            elif graph_networkx.edges[edge]['community'] == 4:
                color_edge_communities.append("red")
            elif graph_networkx.edges[edge]['community'] == 5:
                color_edge_communities.append("purple")
            elif graph_networkx.edges[edge]['community'] == 6:
                color_edge_communities.append("brown")
            elif graph_networkx.edges[edge]['community'] == 7:
                color_edge_communities.append("pink")
            elif graph_networkx.edges[edge]['community'] == 8:
                color_edge_communities.append("grey")
            elif graph_networkx.edges[edge]['community'] == 9:
                color_edge_communities.append("olive")
            
        tuple_internal_color_2 = (tuple_internal_color[0],color_edge_communities)

        dataset_enrich = dataset_with_dummies
        plot_variable_distribution(classifier,pd.Series(variables),scomplex,tuple_internal_color_2,dataset_enrich,continue_features,features_imp)
        
    
    return variables_selected 
    

In [ ]:
binary_features = [] 
categorical_features = (set(dataset_experiment.loc[:, ~dataset_experiment.columns.isin(['Patient_ID','Y','communities'])].columns).difference(set(continue_features))).difference(set(binary_features))

variables_selected, fitted_classifier_models, x = computational_phenotyping(dataset_with_communities,
                                                                            ['Patient_ID','Y','communities'],
                                                                            'random forest',
                                                                            random_seed,
                                                                            list(categorical_features),
                                                                            binary_features,
                                                                            continue_features,
                                                                            dataset_with_communities['Patient_ID'],
                                                                            5,
                                                                            scomplex,
                                                                            tuple_internal_color,
                                                                            True, 
                                                                            G)